In [1]:
import pandas as pd
import numpy as np
import os
from google.colab import drive
from datetime import timedelta
import re

# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Path to the MIMIC-IV dataset
mimic_path = "/content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0"
output_path = '/content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# # Python conversion of patient data SQL files for MIMIC-III
# # Includes: demographics, weight, GCS, Elixhauser score, echo data, hospital times

# def create_demographics(mimic_path, output_path):
#     """
#     Python conversion of demographics.sql for MIMIC-III
#     Retrieves demographic info and static info like patient discharge/death times, mortality flags

#     Parameters:
#     mimic_path (str): Path to MIMIC-III data
#     output_path (str): Path to save the output dataframe

#     Returns:
#     pd.DataFrame: Demographics dataframe
#     """
#     print("Creating demographics dataframe...")

#     # Load necessary tables
#     print("Loading patients, admissions, and icustays tables...")
#     patients = pd.read_csv(f"{mimic_path}/hosp/patients.csv.gz", usecols=['subject_id', 'gender', 'anchor_age', 'anchor_year', 'dod'])
#     admissions = pd.read_csv(f"{mimic_path}/hosp/admissions.csv.gz",
#                              usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
#                                       'admission_type', 'admission_location', 'discharge_location', 'insurance',
#                                       'language', 'marital_status'])
#     icustays = pd.read_csv(f"{mimic_path}/icu/icustays.csv.gz",
#                           usecols=['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'los'])

#     # Convert date columns to datetime
#     date_columns = ['dod', 'admittime', 'dischtime', 'deathtime', 'intime', 'outtime']
#     for df in [patients, admissions, icustays]:
#         for col in date_columns:
#             if col in df.columns:
#                 df[col] = pd.to_datetime(df[col])

#     # Merge tables
#     demo = pd.merge(icustays, admissions, on=['subject_id', 'hadm_id'], how='inner')
#     demo = pd.merge(demo, patients, on='subject_id', how='inner')

#     # Calculate age at admission time using anchor_age and anchor_year
#     demo['birth_year'] = demo['anchor_year'] + demo['anchor_age']

#     # Then calculate age at ICU admission
#     demo['age'] = demo['intime'].dt.year - demo['birth_year']

#     #Adjust age if admission is before birthday that year
#     # This is approximation since we don't have exact DOB
#     demo['age'] = demo['age'] - 0.5
#     demo['age'] = demo['age'].round().astype(int)

#     # Handle age > 89
#     demo.loc[demo['age'] > 89, 'age'] = 91

#     # Create mortality flags
#     demo['hospital_expire_flag'] = (~demo['deathtime'].isna()).astype(int)
#     demo['dod_within_30days'] = ((~demo['dod'].isna()) &
#         (demo['dod'] - demo['outtime']).dt.total_seconds()/(24*3600) <= 30).astype(int)
#     demo['dod_within_90days'] = ((~demo['dod'].isna()) &
#         (demo['dod'] - demo['outtime']).dt.total_seconds()/(24*3600) <= 90).astype(int)
#     demo['dod_within_180days'] = ((~demo['dod'].isna()) &
#         (demo['dod'] - demo['outtime']).dt.total_seconds()/(24*3600) <= 180).astype(int)
#     demo['dod_within_1year'] = ((~demo['dod'].isna()) &
#         (demo['dod'] - demo['outtime']).dt.total_seconds()/(24*3600) <= 365).astype(int)

#     # Select relevant columns
#     demographics = demo[['subject_id', 'hadm_id', 'stay_id', 'gender', 'age',
#         'admission_type', 'admission_location', 'insurance', 'language',
#         'marital_status', 'los',
#         'hospital_expire_flag', 'dod_within_30days', 'dod_within_90days',
#         'dod_within_180days', 'dod_within_1year']]

#     # Save to CSV
#     demographics.to_csv(f"{output_path}/demographics.csv", index=False)
#     print(f"Demographics saved to {output_path}/demographics.csv")

#     return demographics

In [ ]:
# demographics = create_demographics(mimic_path, output_path)

Creating demographics dataframe...
Loading patients, admissions, and icustays tables...
Demographics saved to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/demographics.csv


In [ ]:
import pandas as pd
import numpy as np

def create_demographics2(mimic_path, output_path):
    """
    Implements the SQL query for 'demographics2', computing first admission age, mortality flags, ICU readmission, and merges Elixhauser scores.

    Parameters:
        patients_path (str): CSV with columns ['subject_id','dob','gender','dod']
        admissions_path (str): CSV with ['hadm_id','subject_id','admittime','dischtime','deathtime','hospital_expire_flag']
        icustays_path (str): CSV with ['icustay_id','hadm_id','subject_id','intime','outtime']
        elixhauser_score_path (str): CSV with ['subject_id','hadm_id','elixhauser_vanwalraven']
        output_path (str): CSV path to save the result

    Returns:
        pd.DataFrame: Final demographics2 table
    """
    print("Loading input tables...")
    patients = pd.read_csv(f"{mimic_path}/hosp/patients.csv.gz",
                           usecols=['subject_id', 'gender', 'anchor_age',
                                    'anchor_year', 'dod'],
                           parse_dates=['dod'])
    admissions = pd.read_csv(f"{mimic_path}/hosp/admissions.csv.gz",
                             usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
                                      'admission_type', 'admission_location', 'discharge_location', 'insurance',
                                      'language', 'marital_status', 'hospital_expire_flag'],
                             parse_dates=['admittime','dischtime','deathtime'])
    icustays = pd.read_csv(f"{mimic_path}/icu/icustays.csv.gz",
                          usecols=['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'los'],
                           parse_dates=['intime','outtime'])

    elixhauser = pd.read_csv(f"{output_path}/elixhauser_score.csv",
                             usecols=['subject_id','hadm_id','elixhauser_vanwalraven'])
    print("Input tables loaded.")

    # Join icustays, admissions, and patients
    base = icustays.merge(admissions, on=['hadm_id','subject_id'])
    base = base.merge(patients, on='subject_id')

    # Calculate first admittime and first admit age
    base['first_admittime'] = base.groupby('subject_id')['admittime'].transform('min')
    # current_year =
    # base['first_admit_age'] = ((base['admittime'].dt.date - base['dob'].dt.date).apply(lambda x: x.days) / 365.242).round(2)
    base['birth_year'] = base['anchor_year'] - base['anchor_age']
    # demo['birth_year1'] = demo['anchor_age'] - demo['anchor_year']

    # Then calculate age at ICU admission
    base['first_admit_age'] = base['intime'].dt.year - base['birth_year']

    # #Adjust age if admission is before birthday that year
    # # This is approximation since we don't have exact DOB
    base['first_admit_age'] = base['first_admit_age'] - 0.5
    base['first_admit_age'] = base['first_admit_age'].round().astype(int)

    # Mortality calculations
    base['ICUMort'] = np.where(
        (pd.notnull(base['dod'])) & (base['dod'] > base['intime']) & (base['dod'] < base['outtime']),
        1, 0)
    base['HospMort28day'] = np.where(
        pd.notnull(base['dod']) & (base['dod'] < base['admittime'] + pd.Timedelta(days=28)), 1, 0)
    base['HospMort90day'] = np.where(
        pd.notnull(base['dod']) & (base['dod'] < base['admittime'] + pd.Timedelta(days=90)), 1, 0)

    first_admission_time = base[[
        'subject_id','hadm_id','stay_id','gender','dod','first_admittime','first_admit_age',
        'ICUMort','hospital_expire_flag','HospMort28day','HospMort90day','dischtime','deathtime','intime','outtime'
    ]].copy()

    # HospMort = hospital_expire_flag already present
    first_admission_time = first_admission_time.rename(columns={'hospital_expire_flag':'HospMort'})
    print("First admission time calculated.")

    # ICU readmission: 1 if >1 icustay per subject
    icu_readm = first_admission_time.groupby('subject_id')['stay_id'].count().reset_index()
    icu_readm['ICU_readm'] = np.where(icu_readm['stay_id'] > 1, 1, 0)
    hos_admissions = icu_readm[['subject_id','ICU_readm']]

    # Merge for first_admit_age deidentification, use median 91.4 for >89
    first_admission_time['first_admit_age'] = first_admission_time['first_admit_age'].apply(
        lambda x: 91.4 if x > 89 else x
    )

    # Merge Elixhauser scores
    merged = first_admission_time.merge(hos_admissions, on='subject_id', how='inner')
    merged = merged.merge(elixhauser[['subject_id','hadm_id','elixhauser_vanwalraven']],
                          on=['subject_id','hadm_id'], how='inner')

    # Output columns/rename for compatibility
    outcols = [
        'subject_id', 'hadm_id', 'stay_id',
        'first_admit_age', 'gender', 'ICU_readm',
        'elixhauser_vanwalraven', 'ICUMort', 'HospMort', 'HospMort28day', 'HospMort90day', 'dischtime', 'deathtime'
    ]
    # SQL output column 'elixhauser_score' is 'elixhauser_vanwalraven'

    merged = merged[outcols]
    merged = merged.rename(columns={'elixhauser_vanwalraven':'elixhauser_score'})
    merged = merged.sort_values(['subject_id','hadm_id']).reset_index(drop=True)

    merged.to_csv(f"{output_path}/demographics2.csv", index=False)
    print("Saved demographics2 table to", output_path)
    return merged


In [ ]:
demographics = create_demographics2(mimic_path, output_path)
print(demographics.describe())

In [ ]:
import re
def create_echo_data2(mimic_path, output_path):
    """
    Python conversion of echo-data.sql for MIMIC-III
    Retrieves ECG info, which is also used for weight information

    Parameters:
        mimic_path (str): Path to MIMIC-III data
        output_path (str): Path to save the output dataframe

    Returns:
        pd.DataFrame: Echo data dataframe
    """
    print("Creating echo data dataframe...")

    # Load necessary tables
    print("Loading noteevents table...")

    # In MIMIC-III, echo data is stored in the noteevents table with category 'Echo'
    # Load necessary tavles
    discharge_notes = pd.read_csv(f"{mimic_path}/mimic-iv-note/2.2/note/discharge.csv.gz")
    discharge_detail_notes = pd.read_csv(f"{mimic_path}/mimic-iv-note/2.2/note/discharge_detail.csv.gz")
    radiology_notes = pd.read_csv(f"{mimic_path}/mimic-iv-note/2.2/note/radiology.csv.gz")
    radiology_detail_notes = pd.read_csv(f"{mimic_path}/mimic-iv-note/2.2/note/radiology_detail.csv.gz")


    noteevents = pd.concat([discharge_notes, discharge_detail_notes, radiology_notes, radiology_detail_notes])
    print("Loading noteevents table completes")

    # Filter for Echo notes
    # echo_notes = noteevents[noteevents['category'] == 'Echo'].copy()
    echo_notes = noteevents[noteevents['text'].str.contains('Echo', case=False, na=False)].copy()

    # Extract weight information from text using regex
    # Simplified version; actual regex may need to be more complex


    # Extract 'Indication'
    echo_notes['indication'] = echo_notes['text'].str.extract(r'Indication:\s*(.*?)\n', flags=re.IGNORECASE)

    # Height: strict SQL, then fallback to flexible (both exclude values containing '*')
    height1 = echo_notes["text"].str.extract(r"Height:\s*\(in\)\s*([^\n\*]*)\n", flags=re.IGNORECASE)[0]
    height2 = echo_notes["text"].str.extract(r"height\s*[:=]?\s*([0-9]+\.?\d*)", flags=re.IGNORECASE)[0]
    echo_notes["height"] = pd.to_numeric(height1, errors='coerce')
    missing = echo_notes["height"].isnull()
    echo_notes.loc[missing, "height"] = pd.to_numeric(height2[missing], errors='coerce')
    echo_notes.loc[echo_notes["height"].astype(str).str.contains('\*', na=False), "Height"] = None

    # Weight: strict SQL, then fallback to flexible
    weight1 = echo_notes["text"].str.extract(r"Weight\s*\(lb\):\s*([^\n\*]*)\n", flags=re.IGNORECASE)[0]
    weight2 = echo_notes["text"].str.extract(r"weight\s*[:=]\s*([0-9]+\.?\d*)", flags=re.IGNORECASE)[0]
    echo_notes["weight"] = pd.to_numeric(weight1, errors='coerce')
    missing = echo_notes["weight"].isnull()
    echo_notes.loc[missing, "weight"] = pd.to_numeric(weight2[missing], errors='coerce')
    echo_notes.loc[echo_notes["weight"].astype(str).str.contains('\*', na=False), "Weight"] = None

    # BSA: strict SQL, then fallback flexible
    bsa1 = echo_notes["text"].str.extract(r"BSA\s*\(m2\):\s*([^\s\*]+)", flags=re.IGNORECASE)[0]
    bsa2 = echo_notes["text"].str.extract(r"BSA\s*\(m2\)\s*:?=?\s*([0-9]+\.?\d*)", flags=re.IGNORECASE)[0]
    echo_notes["bsa"] = pd.to_numeric(bsa1, errors='coerce')
    missing = echo_notes["bsa"].isnull()
    echo_notes.loc[missing, "bsa"] = pd.to_numeric(bsa2[missing], errors='coerce')
    echo_notes.loc[echo_notes["bsa"].astype(str).str.contains('\*', na=False), "BSA"] = None

    # BP: SQL, take full field
    echo_notes["bp"] = echo_notes["text"].str.extract(r'BP\s*\(mm Hg\):\s*([^\n]*)', flags=re.IGNORECASE)[0]
    # Systolic/diastolic
    systolic = echo_notes["text"].str.extract(r'BP\s*\(mm Hg\):\s*([0-9]+)\s*/\s*[0-9]+\s*\n', flags=re.IGNORECASE)[0]
    diastolic = echo_notes["text"].str.extract(r'BP\s*\(mm Hg\):\s*[0-9]+\s*/\s*([0-9]+)\s*\n', flags=re.IGNORECASE)[0]
    echo_notes["bp_sys"] = pd.to_numeric(systolic, errors='coerce')
    echo_notes["bp_dias"] = pd.to_numeric(diastolic, errors='coerce')

    # HR: SQL, fallback flexible (exclude *)
    hr1 = echo_notes["text"].str.extract(r'HR\s*\(bpm\):\s*([^\n\*]*)\n', flags=re.IGNORECASE)[0]
    echo_notes["hr"] = pd.to_numeric(hr1, errors='coerce')
    hr2 = echo_notes["text"].str.extract(r'HR\s*\(bpm\)?\s*[:=]?\s*([0-9]+\.?\d*)', flags=re.IGNORECASE)[0]
    missing = echo_notes["hr"].isnull()
    echo_notes.loc[missing, "hr"] = pd.to_numeric(hr2[missing], errors='coerce')
    echo_notes.loc[echo_notes["hr"].astype(str).str.contains('\*', na=False), "HR"] = None

    # Variables mentioned in feedback
    echo_notes['status'] = echo_notes['text'].str.extract(r'Status:\s*(.*?)\n', flags=re.IGNORECASE)
    echo_notes['tissue'] = echo_notes['text'].str.extract(r'Tissue:\s*(.*?)\n', flags=re.IGNORECASE)
    echo_notes['doppler'] = echo_notes['text'].str.extract(r'Doppler:\s*(.*?)\n', flags=re.IGNORECASE)
    echo_notes['contrast'] = echo_notes['text'].str.extract(r'Contrast:\s*(.*?)\n', flags=re.IGNORECASE)
    echo_notes['technical_quality'] = echo_notes['text'].str.extract(r'Technical Quality:\s*(.*?)\n', flags=re.IGNORECASE)


    # echo_notes['weight'] = echo_notes['text'].str.extract(r"weight\s*[:=]\s*(\d{1,4}\.?\d*)", flags=re.IGNORECASE)
    # echo_notes['weight'] = pd.to_numeric(echo_notes['weight'], errors='coerce')


    # Select relevant columns
    echo_data = echo_notes[['subject_id', 'hadm_id', 'charttime', 'indication', 'height', 'weight', 'bsa', 'bp', 'bp_sys',
                            'bp_dias', 'hr', 'status', 'tissue', 'doppler', 'contrast', 'technical_quality']]
    # echo_data.rename(columns={'chartdate': 'charttime'}, inplace=True)

    # Save to CSV
    echo_data.to_csv(f"{output_path}/echo_dat2.csv", index=False)
    print("Echo data saved to output_path/echo_data2.csv")

    return echo_data

<>:47: SyntaxWarning: invalid escape sequence '\*'
<>:55: SyntaxWarning: invalid escape sequence '\*'
<>:63: SyntaxWarning: invalid escape sequence '\*'
<>:79: SyntaxWarning: invalid escape sequence '\*'
<>:47: SyntaxWarning: invalid escape sequence '\*'
<>:55: SyntaxWarning: invalid escape sequence '\*'
<>:63: SyntaxWarning: invalid escape sequence '\*'
<>:79: SyntaxWarning: invalid escape sequence '\*'
/tmp/ipython-input-4221381006.py:47: SyntaxWarning: invalid escape sequence '\*'
  echo_notes.loc[echo_notes["height"].astype(str).str.contains('\*', na=False), "Height"] = None
/tmp/ipython-input-4221381006.py:55: SyntaxWarning: invalid escape sequence '\*'
  echo_notes.loc[echo_notes["weight"].astype(str).str.contains('\*', na=False), "Weight"] = None
/tmp/ipython-input-4221381006.py:63: SyntaxWarning: invalid escape sequence '\*'
  echo_notes.loc[echo_notes["bsa"].astype(str).str.contains('\*', na=False), "BSA"] = None
/tmp/ipython-input-4221381006.py:79: SyntaxWarning: invalid esca

In [ ]:
echo_data = create_echo_data2(mimic_path, output_path)
print(echo_data.head())
print(echo_data.describe())

Creating echo data dataframe...
Loading noteevents table...
Loading noteevents table completes
Echo data saved to output_path/echo_data2.csv
    subject_id     hadm_id            charttime indication  height  weight  \
0     10000032  22595853.0  2180-05-07 00:00:00        NaN     NaN     NaN   
1     10000032  22841357.0  2180-06-27 00:00:00        NaN     NaN     NaN   
9     10000764  27897940.0  2132-10-19 00:00:00        NaN     NaN     NaN   
10    10000826  20032235.0  2146-12-12 00:00:00        NaN     NaN     NaN   
18    10000980  29654838.0  2188-01-05 00:00:00        NaN     NaN     NaN   

    bsa   bp  bp_sys  bp_dias  hr               status tissue doppler  \
0   NaN  NaN     NaN      NaN NaN  Clear and coherent.    NaN     NaN   
1   NaN  NaN     NaN      NaN NaN  Clear and coherent.    NaN     NaN   
9   NaN  NaN     NaN      NaN NaN  Clear and coherent.    NaN     NaN   
10  NaN  NaN     NaN      NaN NaN  Clear and coherent.    NaN     NaN   
18  NaN  NaN     NaN     

In [ ]:
# import re
# def create_echo_data(mimic_path, output_path):
#     """
#     Python conversion of echo-data.sql for MIMIC-III
#     Retrieves ECG info, which is also used for weight information

#     Parameters:
#         mimic_path (str): Path to MIMIC-III data
#         output_path (str): Path to save the output dataframe

#     Returns:
#         pd.DataFrame: Echo data dataframe
#     """
#     print("Creating echo data dataframe...")

#     # Load necessary tables
#     print("Loading noteevents table...")

#     # In MIMIC-III, echo data is stored in the noteevents table with category 'Echo'
#     # Load necessary tavles
#     discharge_notes = pd.read_csv(f"{mimic_path}/mimic-iv-note/2.2/note/discharge.csv.gz")
#     discharge_detail_notes = pd.read_csv(f"{mimic_path}/mimic-iv-note/2.2/note/discharge_detail.csv.gz")
#     radiology_notes = pd.read_csv(f"{mimic_path}/mimic-iv-note/2.2/note/radiology.csv.gz")
#     radiology_detail_notes = pd.read_csv(f"{mimic_path}/mimic-iv-note/2.2/note/radiology_detail.csv.gz")


#     noteevents = pd.concat([discharge_notes, discharge_detail_notes, radiology_notes, radiology_detail_notes])
#     print("Loading noteevents table completes")

#     # Filter for Echo notes
#     # echo_notes = noteevents[noteevents['category'] == 'Echo'].copy()
#     echo_notes = noteevents[noteevents['text'].str.contains('Echo', case=False, na=False)].copy()

#     # Extract weight information from text using regex
#     # Simplified version; actual regex may need to be more complex


#     # Extract 'Indication'
#     echo_notes['indication'] = echo_notes['text'].str.extract(r'Indication:\s*(.*?)\n', flags=re.IGNORECASE)

#     # Height (in inches)
#     echo_notes['height'] = echo_notes['text'].str.extract(r'Height:\s*\(in\)\s*(\d+\.?\d*)', flags=re.IGNORECASE)

#     # Weight (in pounds)
#     echo_notes['weight'] = echo_notes['text'].str.extract(r'Weight\s*\(lb\):\s*(\d+\.?\d*)', flags=re.IGNORECASE)
#     echo_notes['weight'] = pd.to_numeric(echo_notes['weight'], errors='coerce')

#     # BSA (Body Surface Area)
#     echo_notes['bsa'] = echo_notes['text'].str.extract(r'BSA\s*\(m2\):\s*(\d+\.?\d*)', flags=re.IGNORECASE)
#     echo_notes['bsa'] = pd.to_numeric(echo_notes['bsa'], errors='coerce')

#     # Blood Pressure (full string)
#     echo_notes['bp'] = echo_notes['text'].str.extract(r'BP\s*\(mm hg\):\s*(.*)\n', flags=re.IGNORECASE)

#     # Systolic BP
#     echo_notes['bp_sys'] = echo_notes['text'].str.extract(r'BP\s*\(mm hg\):\s*(\d+)/\d+', flags=re.IGNORECASE)
#     echo_notes['bp_sys'] = pd.to_numeric(echo_notes['bp_sys'], errors='coerce')

#     # Diastolic BP
#     echo_notes['bp_dias'] = echo_notes['text'].str.extract(r'BP\s*\(mm hg\):\s*\d+/(\d+)', flags=re.IGNORECASE)
#     echo_notes['bp_dias'] = pd.to_numeric(echo_notes['bp_dias'], errors='coerce')

#     print("Reached half stage")

#     # Heart Rate
#     echo_notes['hr'] = echo_notes['text'].str.extract(r'HR\\s*\(bpm\):\s*(\d+)', flags=re.IGNORECASE)

#     # Variables mentioned in feedback
#     echo_notes['status'] = echo_notes['text'].str.extract(r'Status:\s*(.*?)\n', flags=re.IGNORECASE)
#     echo_notes['tissue'] = echo_notes['text'].str.extract(r'Tissue:\s*(.*?)\n', flags=re.IGNORECASE)
#     echo_notes['doppler'] = echo_notes['text'].str.extract(r'Doppler:\s*(.*?)\n', flags=re.IGNORECASE)
#     echo_notes['contrast'] = echo_notes['text'].str.extract(r'Contrast:\s*(.*?)\n', flags=re.IGNORECASE)
#     echo_notes['technical_quality'] = echo_notes['text'].str.extract(r'Technical Quality:\s*(.*?)\n', flags=re.IGNORECASE)


#     # echo_notes['weight'] = echo_notes['text'].str.extract(r"weight\s*[:=]\s*(\d{1,4}\.?\d*)", flags=re.IGNORECASE)
#     # echo_notes['weight'] = pd.to_numeric(echo_notes['weight'], errors='coerce')


#     # Select relevant columns
#     echo_data = echo_notes[['subject_id', 'hadm_id', 'charttime', 'indication', 'height', 'weight', 'bsa', 'bp', 'bp_sys',
#                             'bp_dias', 'hr', 'status', 'tissue', 'doppler', 'contrast', 'technical_quality']]
#     # echo_data.rename(columns={'chartdate': 'charttime'}, inplace=True)

#     # Save to CSV
#     echo_data.to_csv(f"{output_path}/echo_data.csv", index=False)
#     print("Echo data saved to output_path/echo_data.csv")

#     return echo_data

In [ ]:
# echo_data = create_echo_data(mimic_path, output_path)
# print(echo_data.head())

Creating echo data dataframe...
Loading noteevents table...
Loading noteevents table completes
Reached half stage
Echo data saved to output_path/echo_data.csv
    subject_id     hadm_id            charttime indication height  weight  \
0     10000032  22595853.0  2180-05-07 00:00:00        NaN    NaN     NaN   
1     10000032  22841357.0  2180-06-27 00:00:00        NaN    NaN     NaN   
9     10000764  27897940.0  2132-10-19 00:00:00        NaN    NaN     NaN   
10    10000826  20032235.0  2146-12-12 00:00:00        NaN    NaN     NaN   
18    10000980  29654838.0  2188-01-05 00:00:00        NaN    NaN     NaN   

    bsa   bp  bp_sys  bp_dias   hr               status tissue doppler  \
0   NaN  NaN     NaN      NaN  NaN  Clear and coherent.    NaN     NaN   
1   NaN  NaN     NaN      NaN  NaN  Clear and coherent.    NaN     NaN   
9   NaN  NaN     NaN      NaN  NaN  Clear and coherent.    NaN     NaN   
10  NaN  NaN     NaN      NaN  NaN  Clear and coherent.    NaN     NaN   
18  NaN 

In [ ]:
echo_data.isna().sum()

,0
subject_id,0
hadm_id,106813
charttime,0
indication,109001
height,257349
weight,257349
bsa,256608
bp,256133
bp_sys,256666
bp_dias,256666


In [ ]:
echo_data = create_echo_data(mimic_path, output_path)
print(echo_data.head())

Creating echo data dataframe...
Loading noteevents table...
Echo data saved to output_path/echo_data.csv
    subject_id     hadm_id            charttime  weight
52    10001884  26170293.0  2130-04-19 00:00:00    64.5
63    10002013  27574273.0  2164-03-19 00:00:00    89.0
69    10002013  23581541.0  2160-05-23 00:00:00   210.0
93    10002430  24513842.0  2125-09-30 00:00:00    63.1
97    10002495  24982426.0  2141-05-29 00:00:00    69.0


In [ ]:
echo_data.describe()

,subject_id,hadm_id,BSA
count,2.573490e+05,1.505360e+05,9.000000
mean,1.500761e+07,2.499174e+07,1.991111
std,2.880642e+06,2.883619e+06,0.352188
min,1.000003e+07,2.000003e+07,1.480000
25%,1.250639e+07,2.249855e+07,1.790000
50%,1.500854e+07,2.498672e+07,2.020000
75%,1.749698e+07,2.749371e+07,2.160000
max,1.999999e+07,2.999972e+07,2.660000


In [ ]:
# def create_gcs(mimic_path, output_path):
#     # Python conversion of getGCS.sql for MIMIC-III
#     # Retrieves the GCS eye, motor, verbal, and total score

#     # Parameters:
#     # mimic_path (str): Path to MIMIC-III data
#     # output_path (str): Path to save the output dataframe

#     # Returns:
#     # pd.DataFrame: GCS dataframe

#     print("Creating GCS dataframe...")
#     print("Loading chartevents table...")

#     # In MIMIC-III, GCS scores are stored in the chartevents table
#     chartevents = pd.read_csv(
#         f"{mimic_path}/icu/chartevents.csv.gz",
#         usecols=['subject_id', 'hadm_id', 'stay_id', 'itemid', 'charttime', 'valuenum'],
#         parse_dates=['charttime']
#     )
#      # Filter for GCS related itemids
#     gcs_itemids = {
#         'GCS_Motor': [454, 223901],
#         'GCS_Verbal': [723, 223900],
#         'GCS_Eye': [184, 220739], #223902
#         'GCS_Total': [198, 220994]
#     }

#     all_gcs_itemids = [item for sublist in gcs_itemids.values() for item in sublist]
#     gcs_data = chartevents[chartevents['itemid'].isin(all_gcs_itemids)].copy()

#     # Create columns for each GCS component
#     for component, itemids in gcs_itemids.items():
#         mask = gcs_data['itemid'].isin(itemids)
#         gcs_data.loc[mask, component] = gcs_data.loc[mask, 'valuenum']

#     # Aggregate to get one row per observation
#     gcs = gcs_data.groupby(['subject_id', 'hadm_id', 'stay_id', 'charttime']).agg({
#         'GCS_Motor': 'mean',
#         'GCS_Verbal': 'mean',
#         'GCS_Eye': 'mean',
#         'GCS_Total': 'mean'
#     }).reset_index()

#     # If GCS_Total is missing, calculate it from components
#     mask = gcs['GCS_Total'].isna() & gcs['GCS_Motor'].notna() & gcs['GCS_Verbal'].notna() & gcs['GCS_Eye'].notna()
#     gcs.loc[mask, 'GCS_Total'] = gcs.loc[mask, 'GCS_Motor'] + gcs.loc[mask, 'GCS_Verbal'] + gcs.loc[mask, 'GCS_Eye']

#     # Save to CSV
#     gcs.to_csv(f"{output_path}/gcs.csv", index=False)
#     print(f"GCS data saved to {output_path}/gcs.csv")

#     return gcs

In [ ]:
# gcs = create_gcs(mimic_path, output_path)
# print(gcs.head())

In [ ]:
import pandas as pd
import numpy as np

def create_GCS2(mimic_path, output_path):
    """
    Implements GCS extraction and imputation logic for Metavision only (MIMIC-IV).

    Parameters:
        chartevents_path (str): Path to CHARTEVENTS CSV file (Metavision only).
        output_path (str): Path to save the resulting GCS time series CSV.

    Returns:
        pd.DataFrame: Final GCS DataFrame as per SQL pivoted GCS algorithm.
    """
    print("Loading chartevents...")
    # Only load necessary columns to save RAM
    # usecols = ['stay_id', 'charttime', 'itemid', 'valuenum', 'value', 'error']
    ce = pd.read_csv(
        f"{mimic_path}/icu/chartevents.csv.gz",
        usecols=['subject_id', 'hadm_id', 'stay_id', 'itemid', 'charttime',
                 'value', 'valuenum'],
        parse_dates=['charttime']
    )

    # Filter for Metavision GCS only and no error rows
    metavision_ids = [223900, 223901, 220739]
    ce = ce[ce['itemid'].isin(metavision_ids)]

    # Pivot to GCSMotor, GCSVerbal, GCSEyes columns
    piv = ce.copy()
    piv['GCSMotor'] = np.where(piv['itemid'] == 223901, piv['valuenum'], np.nan)
    piv['GCSVerbal'] = np.where(piv['itemid'] == 223900, piv['valuenum'], np.nan)
    piv['GCSEyes'] = np.where(piv['itemid'] == 220739, piv['valuenum'], np.nan)
    piv['EndoTrachFlag'] = np.where(
        (piv['itemid'] == 223900) & (piv['value'] == 'No Response-ETT'), 1, 0
    )
    # Reduce to necessary columns for grouping
    base = piv.groupby(['subject_id', 'hadm_id', 'stay_id', 'charttime']).agg({
        'GCSMotor': 'max',
        'GCSVerbal': 'max',
        'GCSEyes': 'max',
        'EndoTrachFlag': 'max'
    }).reset_index()

    base = base.sort_values(['subject_id', 'hadm_id','stay_id', 'charttime']).copy()
    base['rn'] = base.groupby('stay_id').cumcount() + 1

    # Provide prior (lookback) values within 6 hours for imputation
    base['charttime'] = pd.to_datetime(base['charttime'])
    base['GCSVerbalPrev'] = base.groupby('stay_id')['GCSVerbal'].shift(1)
    base['GCSMotorPrev'] = base.groupby('stay_id')['GCSMotor'].shift(1)
    base['GCSEyesPrev'] = base.groupby('stay_id')['GCSEyes'].shift(1)
    base['EndoTrachFlagPrev'] = base.groupby('stay_id')['EndoTrachFlag'].shift(1)
    base['charttimePrev'] = base.groupby('stay_id')['charttime'].shift(1)
    # Only carry previous values forward if within 6 hours
    time_diff = (base['charttime'] - base['charttimePrev'])
    within_6h = time_diff < pd.Timedelta(hours=6)
    for c in ['GCSVerbalPrev','GCSMotorPrev','GCSEyesPrev','EndoTrachFlagPrev']:
        base.loc[~within_6h, c] = np.nan

    # Compute GCS, following SQL case logic
    def compute_gcs(row):
        # Sedation—if Verbal is 0, default to 15
        if pd.notnull(row['GCSVerbal']) and row['GCSVerbal'] == 0:
            return 15
        elif pd.isnull(row['GCSVerbal']) and pd.notnull(row['GCSVerbalPrev']) and row['GCSVerbalPrev'] == 0:
            return 15
        # If previous GCS was intubated (0), use current values or default
        elif pd.notnull(row['GCSVerbalPrev']) and row['GCSVerbalPrev'] == 0:
            return (
                (row['GCSMotor'] if pd.notnull(row['GCSMotor']) else 6) +
                (row['GCSVerbal'] if pd.notnull(row['GCSVerbal']) else 5) +
                (row['GCSEyes'] if pd.notnull(row['GCSEyes']) else 4)
            )
        # Otherwise, use current val, or fall back to prev value, or use default
        else:
            gm = row['GCSMotor'] if pd.notnull(row['GCSMotor']) else (
                 row['GCSMotorPrev'] if pd.notnull(row['GCSMotorPrev']) else 6 )
            gv = row['GCSVerbal'] if pd.notnull(row['GCSVerbal']) else (
                 row['GCSVerbalPrev'] if pd.notnull(row['GCSVerbalPrev']) else 5 )
            ge = row['GCSEyes'] if pd.notnull(row['GCSEyes']) else (
                 row['GCSEyesPrev'] if pd.notnull(row['GCSEyesPrev']) else 4 )
            return gm + gv + ge

    base['GCS'] = base.apply(compute_gcs, axis=1)
    base['components_measured'] = (
            base[['GCSMotor','GCSVerbal','GCSEyes']].notnull().sum(axis=1) +
            base[['GCSMotorPrev','GCSVerbalPrev','GCSEyesPrev']].notnull().sum(axis=1)
    )
    # Fill with current or previous values for final reporting
    base['GCSMotor_full'] = base['GCSMotor'].combine_first(base['GCSMotorPrev'])
    base['GCSVerbal_full'] = base['GCSVerbal'].combine_first(base['GCSVerbalPrev'])
    base['GCSEyes_full'] = base['GCSEyes'].combine_first(base['GCSEyesPrev'])
    base['EndoTrachFlag'] = base['EndoTrachFlag'].combine_first(base['EndoTrachFlagPrev'])

    # Priority ordering: prefer complete, non-intubated, lowest GCS as per SQL
    base['priority'] = base['components_measured']*10 + (1-base['EndoTrachFlag']) - base['GCS']
    base = base.sort_values(['subject_id', 'hadm_id','stay_id','charttime','priority'])

    # Select one per icustay_id, charttime (with best quality)
    final = base.groupby(['subject_id', 'hadm_id', 'stay_id','charttime'], as_index=False).first()

    # Output
    results = final[['subject_id', 'hadm_id', 'stay_id','charttime','GCS',
                     'GCSMotor_full','GCSVerbal_full','GCSEyes_full','EndoTrachFlag']]
    results = results.rename(columns={
        'GCSMotor_full':'GCSMotor',
        'GCSVerbal_full':'GCSVerbal',
        'GCSEyes_full':'GCSEyes'
    })

    # Save to CSV
    results = results.sort_values(['stay_id', 'charttime']).reset_index(drop=True)
    results.to_csv(f"{output_path}/gcs2.csv", index=False)
    print(f"GCS result saved to {output_path}")
    return results


In [ ]:
gcs = create_GCS2(mimic_path, output_path)
print(gcs.head())
print(gcs.describe())

In [ ]:
def create_vital_signs2(mimic_path, output_path):
    """
    Python conversion of getVitalsigns.sql for MIMIC-III
    Retrieves vital signs including heart rate, blood pressure, respiratory rate, temperature, and SpO2

    Parameters:
    mimic_path (str): Path to MIMIC-III data
    output_path (str): Path to save the output dataframe

    Returns:
    pd.DataFrame: Vital signs dataframe
    """

    print("Creating vital signs dataframe...")

    # Load necessary tables
    print("Loading chartevents table...")

    # Define vital sign item IDs
    vital_itemids = {
        'HeartRate': [220045],
        'SysBP': [220179, 220050],
        'DiasBP': [220180, 220051],
        'MeanBP': [220052, 220181, 225312],
        'RespRate': [220210, 224690],
        'TempC': [223761, 223762], # Includes both F & C temperatures
        'SpO2' : [220277]
    }

    # Get all item IDs
    all_itemids = []
    for vital_ids in vital_itemids.values():
      all_itemids.extend(vital_ids)

    # Load chartevents table with only the needed columns and item IDs
    # Use chunking to reduce memory usage
    chunk_size = 1000000
    chunks = []

    for chunk in pd.read_csv(f"{mimic_path}/icu/chartevents.csv.gz",
                              usecols=['subject_id', 'hadm_id', 'stay_id', 'itemid', 'charttime', 'valuenum'],
                              parse_dates=['charttime'],
                              chunksize=chunk_size):
        # Filter for relevant item IDs and no errors
        filtered_chunk = chunk[(chunk['itemid'].isin(all_itemids))]

        if not filtered_chunk.empty:
          chunks.append(filtered_chunk)

    if not chunks:
      print("No vital signs data found.")
      return None

    chartevents = pd.concat(chunks, ignore_index=True)

    vitals = pd.DataFrame()
    vitals['subject_id'] = chartevents['subject_id']
    vitals['hadm_id'] = chartevents['hadm_id']
    vitals['stay_id'] = chartevents['stay_id']
    vitals['charttime'] = chartevents['charttime']

    # Process each vital sign

    # Heart Rate
    mask = (
        chartevents['itemid'].isin(vital_itemids['HeartRate']) &
        (chartevents['valuenum'] > 0) &
        (chartevents['valuenum'] < 300)
    )
    vitals.loc[mask, 'HeartRate'] = chartevents.loc[mask, 'valuenum']

    # Systolic BP
    mask = (
        chartevents['itemid'].isin(vital_itemids['SysBP']) &
        (chartevents['valuenum'] > 0) &
        (chartevents['valuenum'] < 400)
    )
    vitals.loc[mask, 'SysBP'] = chartevents.loc[mask, 'valuenum']

    # Diastolic BP
    mask = (
        chartevents['itemid'].isin(vital_itemids['DiasBP']) &
        (chartevents['valuenum'] > 0) &
        (chartevents['valuenum'] < 300)
    )
    vitals.loc[mask, 'DiasBP'] = chartevents.loc[mask, 'valuenum']

    # Mean BP
    mask = (
        chartevents['itemid'].isin(vital_itemids['MeanBP']) &
        (chartevents['valuenum'] > 0) &
        (chartevents['valuenum'] < 300)
    )
    vitals.loc[mask, 'MeanBP'] = chartevents.loc[mask, 'valuenum']

    # Respiratory Rate
    mask = (
        chartevents['itemid'].isin(vital_itemids['RespRate']) &
        (chartevents['valuenum'] > 0) &
        (chartevents['valuenum'] < 70)
    )
    vitals.loc[mask, 'RespRate'] = chartevents.loc[mask, 'valuenum']

    # Temperature - need to convert Fahrenheit to Celsius
    # Fahrenheit temperatures
    mask_f = (
        chartevents['itemid'].isin([223761, 678]) &
        (chartevents['valuenum'] > 70) &
        (chartevents['valuenum'] < 120)
    )
    vitals.loc[mask_f, 'TempC'] = (chartevents.loc[mask_f, 'valuenum'] - 32) / 1.8

    # Celsius temperatures
    mask_c = (
        chartevents['itemid'].isin([223762, 676]) &
        (chartevents['valuenum'] > 10) &
        (chartevents['valuenum'] < 50)
    )
    vitals.loc[mask_c, 'TempC'] = chartevents.loc[mask_c, 'valuenum']

    # SpO2
    mask = (chartevents['itemid'].isin(vital_itemids['SpO2'])) & (chartevents['valuenum'] > 0) & (chartevents['valuenum'] <= 100)
    vitals.loc[mask, 'SpO2'] = chartevents.loc[mask, 'valuenum']

    # # Calculate shock index (Heart Rate / Systolic BP)
    # mask = (vitals['HeartRate'].notna()) & (vitals['SysBP'].notna()) & (vitals['SysBP'] > 0)
    # vitals.loc[mask, 'ShockIndex'] = vitals.loc[mask, 'HeartRate'] / vitals.loc[mask, 'SysBP']

    # Try to merge with GCS data if available
    try:
        gcs = pd.read_csv(f"{output_path}/gcs2.csv", parse_dates=["charttime"])
        # Merge with GCS data
        vitals = vitals.merge(
            gcs[["subject_id", "hadm_id", "stay_id", "charttime", "GCS"]],
            how="left",
            on=["subject_id", "hadm_id", "stay_id", "charttime"]
        )
        vitals.rename(columns={"GCS": "gcs"}, inplace=True)
    except:
        print("GCS data not found. Continuing without GCS.")

    # Group by patient and time to get one row per observation
    vital_signs = vitals.groupby(["subject_id", "hadm_id", "stay_id", "charttime"]).agg({
        'gcs': 'mean',
        'HeartRate': 'mean',
        'SysBP': 'mean',
        'DiasBP': 'mean',
        'MeanBP': 'mean',
        'RespRate': 'mean',
        'TempC': 'mean',
        'SpO2': 'mean',
         }).reset_index()

    # Save to CSV
    vital_signs.to_csv(f"{output_path}/vital_signs2.csv", index=False)
    print(f"Vital signs data saved to {output_path}/vital_signs2.csv")

    return vital_signs

In [ ]:
vital_signs = create_vital_signs2(mimic_path, output_path)
print(vital_signs.head())
print(vital_signs.describe())

Creating vital signs dataframe...
Loading chartevents table...
Vital signs data saved to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/vital_signs2.csv
   subject_id   hadm_id   stay_id           charttime  gcs  HeartRate  SysBP  \
0    10000032  29079034  39553978 2180-07-23 14:00:00  NaN        NaN    NaN   
1    10000032  29079034  39553978 2180-07-23 14:11:00  NaN        NaN   84.0   
2    10000032  29079034  39553978 2180-07-23 14:12:00  NaN       91.0    NaN   
3    10000032  29079034  39553978 2180-07-23 14:13:00  NaN        NaN    NaN   
4    10000032  29079034  39553978 2180-07-23 14:30:00  NaN       93.0   95.0   

   DiasBP  MeanBP  RespRate      TempC  SpO2  
0     NaN     NaN       NaN  37.055556   NaN  
1    48.0    56.0       NaN        NaN   NaN  
2     NaN     NaN      24.0        NaN   NaN  
3     NaN     NaN       NaN        NaN  98.0  
4    59.0    67.0      21.0        NaN  97.0  
         subject_id       hadm_id   

In [ ]:
# vital_signs = create_vital_signs(mimic_path, output_path)
# print(vital_signs.head())

Creating vital signs dataframe...
Loading chartevents table...


/tmp/ipython-input-634399200.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  for chunk in pd.read_csv(f"{mimic_path}/icu/chartevents.csv.gz",


Vital signs data saved to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/vital_signs.csv
   subject_id   hadm_id   stay_id           charttime  HeartRate  SysBP  \
0    10000032  29079034  39553978 2180-07-23 14:00:00        NaN    NaN   
1    10000032  29079034  39553978 2180-07-23 14:11:00        NaN   84.0   
2    10000032  29079034  39553978 2180-07-23 14:12:00       91.0    NaN   
3    10000032  29079034  39553978 2180-07-23 14:13:00        NaN    NaN   
4    10000032  29079034  39553978 2180-07-23 14:30:00       93.0   95.0   

   DiasBP  MeanBP  RespRate      TempC  SpO2  ShockIndex  gcs  
0     NaN     NaN       NaN  37.055556   NaN         NaN  NaN  
1    48.0    56.0       NaN        NaN   NaN         NaN  NaN  
2     NaN     NaN      24.0        NaN   NaN         NaN  NaN  
3     NaN     NaN       NaN        NaN  98.0         NaN  NaN  
4    59.0    67.0      21.0        NaN  97.0         NaN  NaN  


In [ ]:
# Re-run the create_vital_signs function with the corrected code
# vital_signs = create_vital_signs(mimic_path, output_path)
# print(vital_signs.head())

In [ ]:
def create_weight(mimic_path, output_path):
    """
    Python conversion of getweight.sql for MIMIC-III
    Creates a dataframe with weights for patients

    Parameters:
    mimic_path (str): Path to MIMIC-III data
    output_path (str): Path to save the output dataframe

    Returns:
    pd.DataFrame: Weight dataframe
    """
    print("Creating weight dataframe...")

    # Load necessary tables
    print("Loading chartevents and icustays tables...")

    chartevents = pd.read_csv(
        f"{mimic_path}/icu/chartevents.csv.gz",
        usecols=['subject_id', 'hadm_id', 'stay_id', 'itemid', 'charttime', 'valuenum'],
        parse_dates=['charttime']
    )

    icustays = pd.read_csv(
        f"{mimic_path}/icu/icustays.csv.gz",
        usecols=['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime'],
        parse_dates=['intime', 'outtime']
    )

    # Filter for weight measurements
    weight_itemids = [762, 226512, 763, 224639]  # Admit Weight & Daily Weight

    # Filter chartevents for weight measurements
    wt_stg = chartevents[
        (chartevents['itemid'].isin(weight_itemids)) &
        (chartevents['valuenum'].notna()) &
        (chartevents['valuenum'] != 0)
        ].copy()

    # Add weight_type column
    wt_stg['weight_type'] = wt_stg['itemid'].apply(
        lambda x: 'admit' if x in [762, 226512] else 'daily'
    )

    # Assign row numbers within groups
    wt_stg['rn'] = wt_stg.groupby(['stay_id', 'weight_type']).cumcount() + 1

    # Merge with icustays to get intime and outtime
    wt_stg2 = pd.merge(wt_stg, icustays, on=['subject_id', 'hadm_id', 'stay_id'], how='inner')

    # Set starttime based on weight_type
    wt_stg2['starttime'] = wt_stg2.apply(
        lambda row: row['intime'] - timedelta(hours=2) if row['weight_type'] == 'admit' and row['rn'] == 1 else row['charttime'],
        axis=1
    )

    # Filter out rows where weight_type is 'admit' and rn is 1
    wt_stg2 = wt_stg2[~((wt_stg2['weight_type'] == 'admit') & (wt_stg2['rn'] == 1))]

    # Calculate endtime
    wt_stg3 = wt_stg2.sort_values(['stay_id', 'starttime'])
    wt_stg3['endtime'] = wt_stg3.groupby(['stay_id'])['starttime'].shift(-1)
    wt_stg3['endtime'] = wt_stg3['endtime'].fillna(wt_stg3['outtime'] + timedelta(hours=2))

    # Select relevant columns
    wt_stg3 = wt_stg3[['subject_id', 'hadm_id', 'stay_id', 'starttime', 'endtime', 'valuenum']]
    wt_stg3.rename(columns={'valuenum': 'weight'}, inplace=True)

    # Load echo data for additional weight information
    try:
        echo_data = pd.read_csv(f"{output_path}/echo_data.csv", parse_dates=["charttime"])

        # Process echo data for weights
        echo_weights = echo_data[['subject_id', 'hadm_id', 'charttime', 'weight']].dropna(subset=['weight'])
        echo_weights = pd.merge(echo_weights, icustays, on=['subject_id', 'hadm_id'], how='inner')

        # Only include echo weights for patients without weight data
        missing_weights = set(icustays['stay_id']) - set(wt_stg3['stay_id'])
        echo_weights = echo_weights[echo_weights['stay_id'].isin(missing_weights)]

        # Convert pounds to kg if needed (assuming echo weights are in pounds)
        echo_weights['weight'] = echo_weights['weight'] * 0.453592

        # Add echo weights to the weight dataframe
        if not echo_weights.empty:
            echo_weights = echo_weights[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'weight']]
            echo_weights.rename(columns={'charttime': 'starttime'}, inplace=True)
            echo_weights['endtime'] = echo_weights['starttime'] + timedelta(hours=24)  # Assume valid for 24 hours

            # Combine with weight data
            weight_data = pd.concat([wt_stg3, echo_weights[wt_stg3.columns]], ignore_index=True)
        else:
            weight_data = wt_stg3
    except:
        # If echo data is not available, just use the weight data from chartevents
        weight_data = wt_stg3

    # Calculate average weight per patient
    weight_avg = weight_data.groupby(['subject_id', 'hadm_id', 'stay_id'])['weight'].mean().reset_index()

    # Save to CSV
    weight_avg.to_csv(f"{output_path}/weight.csv", index=False)
    print(f"Weight data saved to {output_path}/weight.csv")

    return weight_avg

In [ ]:
weight = create_weight(mimic_path, output_path)
print(weight.head())

Creating weight dataframe...
Loading chartevents and icustays tables...
Weight data saved to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/weight.csv
   subject_id     hadm_id   stay_id  weight
0    10001217  27703517.0  34592300    74.8
1    10001725  25563031.0  31205490    72.5
2    10002013  23581541.0  39060235   104.1
3    10002114  27793700.0  34672098    67.8
4    10002155  23822395.0  33685454    55.0


In [ ]:
import pandas as pd

def create_elixhauser_score(mimic_path, output_path):
    """
    Calculate Elixhauser comorbidities and scores from MIMIC-IV diagnoses_icd and admissions.

    Parameters:
        diagnoses_path (str): Path to diagnoses_icd CSV file (MIMIC-IV)
        admissions_path (str): Path to admissions CSV file (MIMIC-IV)
        output_path (str): Path to save the resulting elixhauser scores CSV

    Returns:
        pd.DataFrame: DataFrame with elixhauser comorbidities and scores per admission
    """

        # Load necessary tables
    print("Loading diagnoses_icd and icustays tables...")
    diagnoses = pd.read_csv(f"{mimic_path}/hosp/diagnoses_icd.csv.gz",
                            usecols=['subject_id', 'hadm_id', 'icd_code', 'icd_version', 'seq_num'])
    admissions = pd.read_csv(f"{mimic_path}/hosp/admissions.csv.gz",
                             usecols=['subject_id', 'hadm_id'])
    print("Loading of diagnoses_icd and icustays tables complete")

    # Filter ICD-9 codes excluding primary diagnosis (seq_num != 1)
    diagnoses_icd9 = diagnoses[(diagnoses['icd_version'] == 9) & (diagnoses['seq_num'] != 1)].copy()

    # Define helper function for substring matching (like SQL SUBSTRING)
    def code_in(icd_code, code_list, length):
        if pd.isna(icd_code):
            return False
        icd_prefix = icd_code[:length]
        return icd_prefix in code_list

    # Apply comorbidity flags (CHF example, repeat for all based on SQL cases)
    def assign_comorbidities(row):
        code = row['icd_code']
        flags = {}

        flags['CHF'] = int(
            code in ['39891','40201','40211','40291','40401','40403','40411','40413','40491','40493']
            or code[:4] in ['4254','4255','4257','4258','4259']
            or code[:3] == '428'
        )

        flags['ARRHY'] = int(
            code in ['42613','42610','42612','99601','99604']
            or code[:4] in ['4260','4267','4269','4270','4271','4272','4273','4274','4276','4278','4279','7850','V450','V533']
        )

        flags['VALVE'] = int(
            code[:4] in ['0932','7463','7464','7465','7466','V422','V433']
            or code[:3] in ['394','395','396','397','424']
        )

        flags['PULMCIRC'] = int(
            code[:4] in ['4150','4151','4170','4178','4179']
            or code[:3] == '416'
        )

        flags['PERIVASC'] = int(
            code[:4] in ['0930','4373','4431','4432','4438','4439','4471','5571','5579','V434']
            or code[:3] in ['440','441']
        )

        flags['HTN'] = int(code[:3] == '401')

        flags['HTNCX'] = int(code[:3] in ['402','403','404','405'])

        flags['PARA'] = int(
            code[:4] in ['3341','3440','3441','3442','3443','3444','3445','3446','3449']
            or code[:3] in ['342','343']
        )

        flags['NEURO'] = int(
            code == '33392'
            or code[:4] in ['3319','3320','3321','3334','3335','3362','3481','3483','7803','7843']
            or code[:3] in ['334','335','340','341','345']
        )

        flags['CHRNLUNG'] = int(
            code[:4] in ['4168','4169','5064','5081','5088']
            or code[:3] in ['490','491','492','493','494','495','496','500','501','502','503','504','505']
        )

        flags['DM'] = int(code[:4] in ['2500','2501','2502','2503'])

        flags['DMCX'] = int(code[:4] in ['2504','2505','2506','2507','2508','2509'])

        flags['HYPOTHY'] = int(
            code[:4] in ['2409','2461','2468']
            or code[:3] in ['243','244']
        )

        flags['RENLFAIL'] = int(
            code in ['40301','40311','40391','40402','40403','40412','40413','40492','40493']
            or code[:4] in ['5880','V420','V451']
            or code[:3] in ['585','586','V56']
        )

        flags['LIVER'] = int(
            code in ['07022','07023','07032','07033','07044','07054']
            or code[:4] in ['0706','0709','4560','4561','4562','5722','5723','5724','5728','5733','5734','5738','5739','V427']
            or code[:3] in ['570','571']
        )

        flags['ULCER'] = int(code[:4] in ['5317','5319','5327','5329','5337','5339','5347','5349'])

        flags['AIDS'] = int(code[:3] in ['042','043','044'])

        flags['LYMPH'] = int(
            code[:4] in ['2030','2386']
            or code[:3] in ['200','201','202']
        )

        flags['METS'] = int(code[:3] in ['196','197','198','199'])

        flags['TUMOR'] = int(
            code[:3] in ['140','141','142','143','144','145','146','147','148','149','150','151','152',
                         '153','154','155','156','157','158','159','160','161','162','163','164','165',
                         '166','167','168','169','170','171','172','174','175','176','177','178','179',
                         '180','181','182','183','184','185','186','187','188','189','190','191','192',
                         '193','194','195']
        )

        flags['ARTH'] = int(
            code in ['72889','72930']
            or code[:4] in ['7010','7100','7101','7102','7103','7104','7108','7109','7112','7193','7285']
            or code[:3] in ['446','714','720','725']
        )

        flags['COAG'] = int(
            code[:4] in ['2871','2873','2874','2875']
            or code[:3] == '286'
        )

        flags['OBESE'] = int(code[:4] == '2780')

        flags['WGHTLOSS'] = int(
            code[:4] in ['7832','7994']
            or code[:3] in ['260','261','262','263']
        )

        flags['LYTES'] = int(
            code[:4] in ['2536']
            or code[:3] == '276'
        )

        flags['BLDLOSS'] = int(code[:4] == '2800')

        flags['ANEMDEF'] = int(
            code[:4] in ['2801','2808','2809']
            or code[:3] == '281'
        )

        flags['ALCOHOL'] = int(
            code[:4] in ['2652','2911','2912','2913','2915','2918','2919','3030','3039','3050','3575','4255','5353','5710','5711','5712','5713','V113']
            or code[:3] == '980'
        )

        flags['DRUG'] = int(
            code == 'V6542'
            or code[:4] in ['3052','3053','3054','3055','3056','3057','3058','3059']
            or code[:3] in ['292','304']
        )

        flags['PSYCH'] = int(
            code in ['29604','29614','29644','29654']
            or code[:4] == '2938'
            or code[:3] in ['295','297','298']
        )

        flags['DEPRESS'] = int(
            code[:4] in ['2962','2963','2965','3004']
            or code[:3] in ['309','311']
        )

        return pd.Series(flags)

    # Assign comorbidities flags per diagnosis
    comorbidity_flags = diagnoses_icd9.apply(assign_comorbidities, axis=1)

    diagnoses_icd9 = pd.concat([diagnoses_icd9[['hadm_id']], comorbidity_flags], axis=1)

    # Aggregate at admission level by max (if any diagnosis for comorbidity =1)
    eligrp = diagnoses_icd9.groupby('hadm_id').max().reset_index()

    # Join with admissions for patient IDs
    adm = admissions[['subject_id', 'hadm_id']]
    elixhauser = adm.merge(eligrp, on='hadm_id', how='left').fillna(0)

    # Create combined Elixhauser flags as per SQL logic
    elixhauser['HYPERTENSION'] = ((elixhauser.get('HTN',0) == 1) | (elixhauser.get('HTNCX',0) ==1 )).astype(int)
    elixhauser['DIABETES_UNCOMPLICATED'] = ((elixhauser.get('DMCX',0) == 1) * 0 + (elixhauser.get('DM',0) ==1)*1).clip(upper=1)
    elixhauser['SOLID_TUMOR'] = ((elixhauser.get('METS',0)==1)*0 + (elixhauser.get('TUMOR',0)==1)*1).clip(upper=1)

    # Calculate van Walraven score
    elixhauser['elixhauser_vanwalraven'] = (
        0 * elixhauser.get('AIDS', 0) +
        0 * elixhauser.get('ALCOHOL', 0) +
        -2 * elixhauser.get('BLDLOSS', 0) +
        7 * elixhauser.get('CHF', 0) +
        3 * elixhauser.get('CHRNLUNG', 0) +
        3 * elixhauser.get('COAG', 0) +
        -2 * elixhauser.get('ANEMDEF', 0) +
        -3 * elixhauser.get('DEPRESS', 0) +
        0 * elixhauser.get('DMCX', 0) +
        0 * elixhauser.get('DM', 0) +
        -7 * elixhauser.get('DRUG', 0) +
        5 * elixhauser.get('LYTES', 0) +
        0 * elixhauser.get('HYPERTENSION', 0) +
        0 * elixhauser.get('HYPOTHY', 0) +
        11 * elixhauser.get('LIVER', 0) +
        9 * elixhauser.get('LYMPH', 0) +
        12 * elixhauser.get('METS', 0) +
        6 * elixhauser.get('NEURO', 0) +
        -4 * elixhauser.get('OBESE', 0) +
        7 * elixhauser.get('PARA', 0) +
        2 * elixhauser.get('PERIVASC', 0) +
        0 * elixhauser.get('ULCER', 0) +
        0 * elixhauser.get('PSYCH', 0) +
        4 * elixhauser.get('PULMCIRC', 0) +
        0 * elixhauser.get('ARTH', 0) +
        5 * elixhauser.get('RENLFAIL', 0) +
        4 * elixhauser.get('SOLID_TUMOR', 0) +
        -1 * elixhauser.get('VALVE', 0) +
        6 * elixhauser.get('WGHTLOSS', 0)
    )

    # SID29 Score
    elixhauser['elixhauser_SID29'] = (
        0 * elixhauser.get('AIDS', 0) +
        -2 * elixhauser.get('ALCOHOL', 0) +
        -2 * elixhauser.get('BLDLOSS', 0) +
        9 * elixhauser.get('CHF', 0) +
        3 * elixhauser.get('CHRNLUNG', 0) +
        9 * elixhauser.get('COAG', 0) +
        0 * elixhauser.get('ANEMDEF', 0) +
        -4 * elixhauser.get('DEPRESS', 0) +
        0 * elixhauser.get('DMCX', 0) +
        -1 * elixhauser.get('DM', 0) +
        -8 * elixhauser.get('DRUG', 0) +
        9 * elixhauser.get('LYTES', 0) +
        -1 * elixhauser.get('HTN', 0) +
        0 * elixhauser.get('HYPOTHY', 0) +
        5 * elixhauser.get('LIVER', 0) +
        6 * elixhauser.get('LYMPH', 0) +
        13 * elixhauser.get('METS', 0) +
        4 * elixhauser.get('NEURO', 0) +
        -4 * elixhauser.get('OBESE', 0) +
        3 * elixhauser.get('PARA', 0) +
        0 * elixhauser.get('ULCER', 0) +
        4 * elixhauser.get('PERIVASC', 0) +
        -4 * elixhauser.get('PSYCH', 0) +
        5 * elixhauser.get('PULMCIRC', 0) +
        6 * elixhauser.get('RENLFAIL', 0) +
        0 * elixhauser.get('ARTH', 0) +
        8 * elixhauser.get('SOLID_TUMOR', 0) +
        0 * elixhauser.get('VALVE', 0) +
        8 * elixhauser.get('WGHTLOSS', 0)
    )

    # SID30 Score
    elixhauser['elixhauser_SID30'] = (
        0 * elixhauser.get('AIDS', 0) +
        0 * elixhauser.get('ALCOHOL', 0) +
        -3 * elixhauser.get('BLDLOSS', 0) +
        8 * elixhauser.get('ARRHY', 0) +
        9 * elixhauser.get('CHF', 0) +
        3 * elixhauser.get('CHRNLUNG', 0) +
        12 * elixhauser.get('COAG', 0) +
        0 * elixhauser.get('ANEMDEF', 0) +
        -5 * elixhauser.get('DEPRESS', 0) +
        1 * elixhauser.get('DMCX', 0) +
        0 * elixhauser.get('DM', 0) +
        -11 * elixhauser.get('DRUG', 0) +
        11 * elixhauser.get('LYTES', 0) +
        -2 * elixhauser.get('HTN', 0) +
        0 * elixhauser.get('HYPOTHY', 0) +
        7 * elixhauser.get('LIVER', 0) +
        8 * elixhauser.get('LYMPH', 0) +
        17 * elixhauser.get('METS', 0) +
        5 * elixhauser.get('NEURO', 0) +
        -5 * elixhauser.get('OBESE', 0) +
        4 * elixhauser.get('PARA', 0) +
        0 * elixhauser.get('ULCER', 0) +
        4 * elixhauser.get('PERIVASC', 0) +
        -6 * elixhauser.get('PSYCH', 0) +
        5 * elixhauser.get('PULMCIRC', 0) +
        7 * elixhauser.get('RENLFAIL', 0) +
        0 * elixhauser.get('ARTH', 0) +
        10 * elixhauser.get('SOLID_TUMOR', 0) +
        0 * elixhauser.get('VALVE', 0) +
        10 * elixhauser.get('WGHTLOSS', 0)
    )

    # Sort output by subject_id, hadm_id
    elixhauser = elixhauser.sort_values(['subject_id', 'hadm_id']).reset_index(drop=True)

    # Save to CSV
    elixhauser.to_csv(f"{output_path}/elixhauser_score.csv", index=False)
    print(f"Elixhauser score saved to {output_path}/elixhauser_score.csv")

    return elixhauser


In [ ]:
elixhauser_score = create_elixhauser_score(mimic_path, output_path)
print(elixhauser_score.head())

Loading diagnoses_icd and icustays tables...
Loading of diagnoses_icd and icustays tables complete
Elixhauser score saved to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/elixhauser_score.csv
   subject_id   hadm_id  CHF  ARRHY  VALVE  PULMCIRC  PERIVASC  HTN  HTNCX  \
0    10000032  22595853  0.0    0.0    0.0       0.0       0.0  0.0    0.0   
1    10000032  22841357  0.0    0.0    0.0       0.0       0.0  0.0    0.0   
2    10000032  25742920  0.0    0.0    0.0       0.0       0.0  0.0    0.0   
3    10000032  29079034  0.0    0.0    0.0       0.0       0.0  0.0    0.0   
4    10000068  25022803  0.0    0.0    0.0       0.0       0.0  0.0    0.0   

   PARA  ...  ALCOHOL  DRUG  PSYCH  DEPRESS  HYPERTENSION  \
0   0.0  ...      0.0   0.0    0.0      1.0             0   
1   0.0  ...      0.0   0.0    0.0      0.0             0   
2   0.0  ...      0.0   0.0    0.0      0.0             0   
3   0.0  ...      0.0   0.0    0.0      0.0   

In [ ]:
elixhauser_score.describe()

,subject_id,hadm_id,CHF,ARRHY,VALVE,PULMCIRC,PERIVASC,HTN,HTNCX,PARA,...,ALCOHOL,DRUG,PSYCH,DEPRESS,HYPERTENSION,DIABETES_UNCOMPLICATED,SOLID_TUMOR,elixhauser_vanwalraven,elixhauser_SID29,elixhauser_SID30
count,5.460280e+05,5.460280e+05,546028.000000,546028.000000,546028.000000,546028.000000,546028.00000,546028.000000,546028.000000,546028.000000,...,546028.000000,546028.000000,546028.000000,546028.000000,546028.000000,546028.000000,546028.00000,546028.000000,546028.000000,546028.000000
mean,1.501118e+07,2.500100e+07,0.071731,0.096222,0.032388,0.017891,0.02918,0.193448,0.061836,0.007417,...,0.039465,0.025387,0.008992,0.081659,0.255046,0.086356,0.02473,2.415074,2.557807,3.757725
std,2.877694e+06,2.888710e+06,0.258042,0.294896,0.177030,0.132556,0.16831,0.395001,0.240857,0.085803,...,0.194699,0.157298,0.094400,0.273844,0.435887,0.280890,0.15530,5.884100,7.113341,9.602229
min,1.000003e+07,2.000002e+07,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,-17.000000,-24.000000,-29.000000
25%,1.252380e+07,2.249662e+07,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
50%,1.501961e+07,2.500385e+07,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
75%,1.750403e+07,2.750282e+07,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.00000,3.000000,2.000000,3.000000
max,1.999999e+07,2.999994e+07,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,62.000000,70.000000,94.000000


In [ ]:
def create_hosp_mort_and_in_out_times(mimic_path, output_path):
    """
    Python conversion of hospmortandinouttimes.sql for MIMIC-III
    Retrieves IN and OUT times of ICU stays

    Parameters:
    mimic_path (str): Path to MIMIC-III data
    output_path (str): Path to save the output dataframe

    Returns:
    pd.DataFrame: Hospital mortality and in/out times dataframe
    """

    print("Creating hospital mortality and in/out times dataframe...")

    # Load necessary tables
    print("Loading patients, admissions, and icustays tables...")
    patients = pd.read_csv(f"{mimic_path}/hosp/patients.csv.gz", usecols=['subject_id', 'dod'])
    admissions = pd.read_csv(f"{mimic_path}/hosp/admissions.csv.gz",
                             usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'hospital_expire_flag'])
    icustays = pd.read_csv(f"{mimic_path}/icu/icustays.csv.gz",
                           usecols=['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'los'])

    # Convert date columns to datetime
    date_columns = ['dod', 'admittime', 'dischtime', 'deathtime', 'intime', 'outtime']
    for df in [patients, admissions, icustays]:
        for col in date_columns:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col])

    # Merge tables
    hosp_mort = pd.merge(icustays, admissions, on=['subject_id', 'hadm_id'], how='inner')
    hosp_mort = pd.merge(hosp_mort, patients, on='subject_id', how='inner')

    # Select relevant columns
    hosp_mort_and_times = hosp_mort[['subject_id', 'hadm_id', 'stay_id',
                                     'intime', 'outtime', 'admittime', 'dischtime',
                                     'deathtime', 'hospital_expire_flag', 'dod']]
    # Save to CSV
    hosp_mort_and_times.to_csv(f"{output_path}/hosp_mort_and_in_out_times.csv", index=False)
    print(f"Hospital mortality and in/out times saved to {output_path}/hosp_mort_and_in_out_times.csv")
    return hosp_mort_and_times

In [ ]:
hosp_mort_and_in_out_times = create_hosp_mort_and_in_out_times(mimic_path, output_path)
print(hosp_mort_and_in_out_times.head())

Creating hospital mortality and in/out times dataframe...
Loading patients, admissions, and icustays tables...
Hospital mortality and in/out times saved to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/hosp_mort_and_in_out_times.csv
   subject_id   hadm_id   stay_id              intime             outtime  \
0    10000032  29079034  39553978 2180-07-23 14:00:00 2180-07-23 23:50:47   
1    10000690  25860671  37081114 2150-11-02 19:37:00 2150-11-06 17:03:17   
2    10000980  26913865  39765666 2189-06-27 08:42:00 2189-06-27 20:38:27   
3    10001217  24597018  37067082 2157-11-20 19:18:02 2157-11-21 22:08:00   
4    10001217  27703517  34592300 2157-12-19 15:42:24 2157-12-20 14:27:41   

            admittime           dischtime deathtime  hospital_expire_flag  \
0 2180-07-23 12:35:00 2180-07-25 17:55:00       NaT                     0   
1 2150-11-02 18:02:00 2150-11-12 13:45:00       NaT                     0   
2 2189-06-27 07:38:00 21